# Lecture Bot (skeleton)

Ask questions about the lecture. The bot does **no retrieval (RAG)**: every document in
`docs/` is converted to text and pasted into the system prompt in full. You can print that
prompt and see exactly what the model knows.

**Placeholder:** `docs/` currently holds the syllabus (`Lehreinheiten.pdf`) and the module
description. Drop the real lecture PDFs into `docs/` and re-run from the top.

In [ ]:
from pathlib import Path
import subprocess

import ollama

MODEL = "qwen2.5:7b"
NUM_CTX = 8192          # context window in tokens: documents + chat history must fit
DOCS_DIR = Path("docs")

## 1. Load the documents

`pdftotext` extracts the plain text. A token is roughly 4 characters of English or German
text, so `chars / 4` is a quick size estimate before we ask the model.

In [ ]:
def pdf_to_text(path):
    return subprocess.run(["pdftotext", "-layout", str(path), "-"],
                          capture_output=True, text=True, check=True).stdout

docs = {p.name: pdf_to_text(p) for p in sorted(DOCS_DIR.glob("*.pdf"))}

for name, text in docs.items():
    print(f"{name:40s} {len(text.split()):6d} words  ~{len(text) // 4:6d} tokens")
total = sum(len(t) for t in docs.values()) // 4
print(f"{'total':40s} {'':6s}        ~{total:6d} tokens of {NUM_CTX}")

## 2. Build the system prompt

This is the whole trick: instructions plus the raw document text. Print it and read it.

In [ ]:
INSTRUCTIONS = """You are a teaching assistant for the module "Agentic AI".
Answer only from the lecture material below. If the material does not cover the
question, say so. Name the document you used. Answer in the language of the question."""

SYSTEM_PROMPT = INSTRUCTIONS + "\n\n" + "\n\n".join(
    f"=== {name} ===\n{text}" for name, text in docs.items()
)

print(SYSTEM_PROMPT)

## 3. Ask

`history` keeps the conversation, so follow-up questions work. After each answer we print
how many tokens the model actually read (`prompt_eval_count`). Ollama silently cuts off
anything beyond `NUM_CTX`, so if that number gets close to the limit, call `reset()`.

In [ ]:
history = []

def reset():
    history.clear()

def ask(question):
    history.append({"role": "user", "content": question})
    response = ollama.chat(
        model=MODEL,
        messages=[{"role": "system", "content": SYSTEM_PROMPT}] + history,
        options={"num_ctx": NUM_CTX, "temperature": 0},
    )
    answer = response.message.content
    history.append({"role": "assistant", "content": answer})
    print(answer)
    print(f"\n[{response.prompt_eval_count} of {NUM_CTX} context tokens used]")

In [ ]:
ask("Wann wird MCP behandelt und worum geht es dabei?")

In [ ]:
ask("Und in welcher Woche kommen danach die Reasoning Models?")

In [ ]:
# Not in the material: the bot should say so instead of inventing an answer.
ask("Wie heißt der Hund des Professors?")